# Lab 02 - Model Architecture, Size, and Memory

**Chapter 2 (Understanding Foundation Models): Training Data, Modeling, Model Size** | Environment: **cluster** (your Docker JupyterLab) | Time: about 4 hours

At Larkspur Ridge IT you are the AI engineer building the help-desk assistant. Before you choose a model, you need to answer two practical questions the chapter keeps returning to: *how big is this model really*, and *what does that size cost me in memory and speed?* In Part A you read the real configuration files of the four course models and do the memory math by hand. In Part B you measure how fast three of them actually generate text on the course cluster and connect the speed to the number of **active** parameters.

**Before you start**
- Connect to the **KU VPN**, run `docker compose up`, and open this notebook in JupyterLab (http://localhost:8888).
- Your `notebooks/00_setup_check.ipynb` from Lab 01 should be all green.
- Use **Run > Run All Cells** once at the start to see where the TODOs are. Then work top to bottom.

## Setup
Run this cell. It connects to the course gateway and creates your self-checker `ck`.

In [ ]:
import json, math, time, statistics
from pathlib import Path

import requests
import pandas as pd
import matplotlib.pyplot as plt

from itec632 import get_client, MODELS, NO_THINK
from itec632.grading import LabChecker

ck = LabChecker("lab02", part="cluster")
client = get_client()

available = {m.id for m in client.models.list().data}
print("Models visible on the gateway:", sorted(available))
ck.check("setup_models_visible",
         {MODELS["main"], MODELS["dense"], MODELS["small"]} <= available, 1,
         "Connect to the KU VPN and re-run this cell. If it stays red, run notebooks/00_setup_check.ipynb.");

# Part A - Guided practice: reading a model by its config

Every open-weight model on Hugging Face ships a `config.json`. It is the blueprint the chapter describes in **"Modeling"**: how many transformer blocks, how wide each one is, how many attention heads, and (for a mixture-of-experts model) how many experts. With it you can estimate parameter counts and memory without downloading a single weight.

The course gateway serves four chat models. Their Hugging Face repositories are below. They are all newer than the models in the book, but the ideas are the same.

| Gateway alias | Hugging Face repo | Type |
|---|---|---|
| `qwen-moe` | `Qwen/Qwen3.6-35B-A3B` | Mixture of experts (MoE) |
| `qwen-dense` | `Qwen/Qwen3.8-27B` | Dense |
| `gemma-moe` | `google/gemma-4-26B-A4B-it` | MoE, different family |
| `qwen-small` | `Qwen/Qwen3.5-4B` | Dense, small |

## A.1 Download the configs

The next cell downloads each `config.json` over plain HTTPS. If the download fails (no internet, Hugging Face down), it falls back to a snapshot saved in this lab's `model_configs/` folder, so you can keep working either way. These models are multimodal, so the file has a `text_config` section for the language model; we use that part.

In [ ]:
REPOS = {
    "qwen-moe":   "Qwen/Qwen3.6-35B-A3B",
    "qwen-dense": "Qwen/Qwen3.8-27B",
    "gemma-moe":  "google/gemma-4-26B-A4B-it",
    "qwen-small": "Qwen/Qwen3.5-4B",
}
SNAPSHOT_DIR = Path("model_configs")   # bundled with this lab


def load_config(alias):
    """Return (text_config dict, source string) for one gateway alias."""
    url = f"https://huggingface.co/{REPOS[alias]}/resolve/main/config.json"
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        cfg, source = r.json(), "huggingface.co"
    except Exception as e:  # no internet, blocked, or Hugging Face down
        path = SNAPSHOT_DIR / f"{alias}.json"
        if not path.exists():
            raise FileNotFoundError(f"Download failed ({type(e).__name__}) and {path} is missing. "
                                    "Open this notebook from inside labs/lab02-model-size/.") from e
        cfg, source = json.loads(path.read_text()), f"snapshot {path}"
    return cfg.get("text_config", cfg), source


configs = {}
for alias in REPOS:
    configs[alias], src = load_config(alias)
    print(f"{alias:11} <- {src}")

FIELDS = ["hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim",
          "intermediate_size", "num_experts", "num_experts_per_tok", "top_k_experts",
          "moe_intermediate_size", "vocab_size", "tie_word_embeddings"]
table = pd.DataFrame({a: {f: c.get(f, "-") for f in FIELDS} for a, c in configs.items()})
for a, c in configs.items():
    lt = c.get("layer_types", [])
    table.loc["layer types", a] = ", ".join(f"{t}: {lt.count(t)}" for t in sorted(set(lt))) or "-"
table

Look at the table before moving on. A few things to notice:

- `qwen-moe` is *narrower* (`hidden_size` 2048) than `qwen-dense` (5120), yet it has more total parameters. The extra parameters live in its **experts**.
- The two MoE configs name the same idea differently: Qwen says `num_experts_per_tok`, Gemma says `top_k_experts`. Reading configs across model families always involves a little translation.
- Most layers in the Qwen models are `linear_attention` (a recurrent-style layer called Gated DeltaNet), and only one in four is ordinary `full_attention`. Gemma mixes `sliding_attention` (looks back only 1,024 tokens) with a few `full_attention` layers. Chapter 2's **"Other model architectures"** discussion (RWKV, Mamba, Jamba) is about exactly this kind of design: keep the quality of attention while avoiding its cost on long inputs. You'll see the payoff in A.4.

**TODO A.1:** complete `moe_info(cfg)` so it works for both families.

In [ ]:
def moe_info(cfg):
    """Return (num_experts, experts_used_per_token). Dense models return (0, 0).

    Qwen MoE configs use 'num_experts' and 'num_experts_per_tok'.
    Gemma 4 configs use 'num_experts' and 'top_k_experts'.
    Example: moe_info(configs["qwen-moe"]) -> (256, 8)
    """
    # TODO: read the two numbers with cfg.get(...); use 0 when a key is missing
    num_experts = None
    per_token = None
    return num_experts, per_token


for a in configs:
    print(f"{a:11} experts (total, per token):", moe_info(configs[a]))

In [ ]:
ck.check("A1_moe_info_qwen", moe_info(configs["qwen-moe"]) == (256, 8), 2,
         "qwen-moe should give (256, 8). Use cfg.get('num_experts', 0) and cfg.get('num_experts_per_tok', ...).")
ck.check("A1_moe_info_gemma_and_dense",
         moe_info(configs["gemma-moe"]) == (128, 8) and moe_info(configs["qwen-dense"]) == (0, 0), 2,
         "Gemma uses the key 'top_k_experts'. Dense models have neither key, so return (0, 0).");

## A.2 Count the parameters

A transformer block has two big parts: **attention** and an **MLP** module (Chapter 2, **"Transformer architecture"**; its linear layers are also called feedforward layers). In most models the MLP holds the majority of the parameters, as you will see in your own counts. Modern models use a "gated" MLP with **three** weight matrices (gate, up, down), each of size `hidden_size x intermediate_size`, so:

`MLP parameters = 3 x hidden_size x intermediate_size`

In a **mixture of experts** model, each expert is just a small MLP. The router picks a few experts per token. So a MoE layer has two numbers:
- **total** parameters = all experts (you must store all of them in memory)
- **active** parameters = only the experts used for one token (what you compute with)

The chapter's MoE example (section **"Model Size"**) is Mixtral 8x7B: 46.7B parameters in total, but only 12.9B active per token, so it runs at roughly the cost of a 12.9B model.

**TODO A.2:** write `mlp_params`. The helper cell after it uses your function (and your `moe_info`) to count everything else for you: attention, embeddings, and the special layer types.

In [ ]:
def mlp_params(hidden_size, intermediate_size):
    """Parameters in one gated MLP (gate + up + down matrices, no biases).
    Example: mlp_params(10, 20) -> 600
    """
    # TODO: return 3 * hidden_size * intermediate_size
    return None


print(mlp_params(10, 20))

Run the helper below (no changes needed). Read the comments if you are curious how attention layers are counted; you do not need to modify it.

In [ ]:
# ---- Helper: full parameter count from a config (just run) ----
def attention_params(cfg):
    """Parameters in all attention-type layers (approximate; ignores small norms and biases)."""
    h, total = cfg["hidden_size"], 0
    for lt in cfg.get("layer_types") or ["full_attention"] * cfg["num_hidden_layers"]:
        if lt == "linear_attention":          # Qwen's Gated DeltaNet layers
            kd = cfg["linear_num_key_heads"] * cfg["linear_key_head_dim"]
            vd = cfg["linear_num_value_heads"] * cfg["linear_value_head_dim"]
            total += (h * (2 * kd + 2 * vd) + h * 2 * cfg["linear_num_value_heads"]
                      + (2 * kd + vd) * cfg["linear_conv_kernel_dim"] + vd * h)
            continue
        nh = cfg["num_attention_heads"]
        if lt == "full_attention" and "global_head_dim" in cfg:   # Gemma 4 global layers
            hd = cfg["global_head_dim"]
            nkv = cfg.get("num_global_key_value_heads", cfg["num_key_value_heads"])
            kv_mats = 1 if cfg.get("attention_k_eq_v") else 2      # keys and values share weights
        else:                                                     # ordinary / sliding attention
            hd, nkv, kv_mats = cfg.get("head_dim", h // nh), cfg["num_key_value_heads"], 2
        q_out = nh * hd * (2 if cfg.get("attn_output_gate") else 1)
        total += h * q_out + kv_mats * h * nkv * hd + nh * hd * h      # Q (+gate), K, V, output
    return total


def mlp_layer_params(cfg):
    """(total, active) MLP parameters in ONE layer, using your mlp_params and moe_info."""
    h = cfg["hidden_size"]
    n_exp, k = moe_info(cfg)
    if not n_exp:
        p = mlp_params(h, cfg["intermediate_size"])
        return p, p
    expert = mlp_params(h, cfg["moe_intermediate_size"])
    always_on = 0
    if cfg.get("shared_expert_intermediate_size"):                       # Qwen: one shared expert
        always_on += mlp_params(h, cfg["shared_expert_intermediate_size"])
    if cfg.get("enable_moe_block") and cfg.get("intermediate_size"):     # Gemma 4: dense MLP beside experts
        always_on += mlp_params(h, cfg["intermediate_size"])
    router = h * n_exp
    return n_exp * expert + always_on + router, k * expert + always_on + router


def count_params(cfg):
    """Dictionary of parameter counts in billions."""
    L, h, V = cfg["num_hidden_layers"], cfg["hidden_size"], cfg["vocab_size"]
    emb = V * h * (1 if cfg.get("tie_word_embeddings") else 2)   # input embeddings (+ output head if untied)
    attn = attention_params(cfg)
    mlp_t, mlp_a = mlp_layer_params(cfg)
    norms = 2 * L * h
    return {"total_B": (emb + attn + L * mlp_t + norms) / 1e9,
            "active_B": (emb + attn + L * mlp_a + norms) / 1e9,
            "embedding_B": emb / 1e9, "attention_B": attn / 1e9, "mlp_total_B": L * mlp_t / 1e9}


try:
    params = pd.DataFrame({a: count_params(c) for a, c in configs.items()}).T.round(2)
    display(params)
except (TypeError, KeyError):
    params = None
    print("Complete the TODOs in A.1 and A.2 first (moe_info or mlp_params still returns None).")

Compare your counts with the names: 35B-**A3B** means about 35B total and 3B active; Gemma's model card says 25.2B total and 3.8B active. Your "active" number for `qwen-moe` will come out a little above 3B because we counted the whole embedding table as active. Looking up one row of a table is almost free, so many vendors leave embeddings out of the "active" number. Parameter counts always depend on such counting conventions.

**TODO A.2 (continued):** compute the active fraction for each MoE model.

In [ ]:
# TODO: active_fraction = active_B / total_B for each MoE model (use the `params` table above)
active_fraction = {
    "qwen-moe": None,
    "gemma-moe": None,
}
print(active_fraction)

In [ ]:
ok_counts = params is not None
if ok_counts:
    P = params
    ck.record("qwen_moe_total_B", float(P.loc["qwen-moe", "total_B"]))
    ck.record("qwen_moe_active_B", float(P.loc["qwen-moe", "active_B"]))
    ck.record("qwen_dense_total_B", float(P.loc["qwen-dense", "total_B"]))
ck.check("A2_mlp_params", mlp_params(10, 20) == 600, 1, "mlp_params(10, 20) should be 600 (3 x 10 x 20).")
ck.check("A2_dense_counts_plausible", ok_counts and 25 <= P.loc["qwen-dense", "total_B"] <= 29
         and 3.6 <= P.loc["qwen-small", "total_B"] <= 4.8, 2,
         "qwen-dense should land near 27B and qwen-small near 4B. Check mlp_params.")
ck.check("A2_moe_counts_plausible", ok_counts and 32 <= P.loc["qwen-moe", "total_B"] <= 37
         and 2.5 <= P.loc["qwen-moe", "active_B"] <= 4.2 and 23 <= P.loc["gemma-moe", "total_B"] <= 27, 2,
         "qwen-moe should be ~35B total / ~3-4B active; gemma-moe ~25B total. Check moe_info.")
af = active_fraction.get("qwen-moe")
ck.check("A2_active_fraction", af is not None and 0.05 <= af <= 0.15, 1,
         "Divide active_B by total_B for qwen-moe; it should be roughly 0.1.");

## A.3 How much memory do the weights need?

Chapter 2's **"Model Size"** section gives the rule of thumb: memory for the weights = number of parameters x bytes per parameter. A 7B model in 16-bit (2 bytes) needs at least 14 GB just to hold its weights. Quantization (Chapter 7) stores weights in fewer bits: 8-bit is 1 byte per parameter, 4-bit is half a byte.

**TODO A.3:** write `weight_gb`. Use decimal gigabytes (1 GB = 1e9 bytes).

In [ ]:
def weight_gb(params_billion, bits):
    """Memory for the weights in GB. Example: weight_gb(7, 16) -> 14.0"""
    # TODO: params (in billions) * 1e9 * bits / 8, converted to GB (divide by 1e9)
    return None


print(weight_gb(7, 16))

In [ ]:
mem = None
if params is not None and weight_gb(7, 16) is not None:
    mem = pd.DataFrame({
        "total_B": params["total_B"],
        "bf16_GB": params["total_B"].apply(lambda p: weight_gb(p, 16)),
        "8bit_GB": params["total_B"].apply(lambda p: weight_gb(p, 8)),
        "4bit_GB": params["total_B"].apply(lambda p: weight_gb(p, 4)),
    }).round(1)
    display(mem)
    print("Course cluster: each Mac Studio M3 Ultra has 96 GB of unified memory (about 72 GB usable by the GPU by default).")
    print("Free Colab T4: 16 GB of GPU memory, about 15 GB usable.")
else:
    print("Complete the TODOs above first.")

In [ ]:
ck.check("A3_weight_gb", weight_gb(7, 16) == 14.0 and weight_gb(10, 4) == 5.0, 2,
         "weight_gb(7, 16) should be 14.0 and weight_gb(10, 4) should be 5.0.")
ck.check("A3_memory_table", mem is not None and 60 <= mem.loc["qwen-moe", "bf16_GB"] <= 75, 1,
         "The bf16 row for qwen-moe should be about 69 GB. Re-run the table cell after fixing weight_gb.")
if mem is not None:
    ck.record("qwen_moe_8bit_GB", float(mem.loc["qwen-moe", "8bit_GB"]))
    ck.record("qwen_dense_4bit_GB", float(mem.loc["qwen-dense", "4bit_GB"]))

Now connect this to the course cluster. The instructor runs `qwen-moe` at 8-bit on two Mac Studios (one copy each, load-balanced) and puts `qwen-dense` at 4-bit together with `gemma-moe`, `qwen-small`, and the embedding and reranker models on the third box. Use your table to check that this fits under about 72 GB per box. You will justify (or criticize) this layout in the analysis.

## A.4 The KV cache: memory that grows with the conversation

Weights are not the only memory cost. While generating, a transformer keeps the **keys and values** of every previous token for every attention layer, so it does not recompute them. This "KV cache" grows with context length (Chapter 9 returns to it in detail). For one token:

`KV bytes per token = 2 (K and V) x layers x kv_heads x head_dim x bytes_per_value`

Attention heads share key/value heads ("grouped-query attention"), which is why the config lists far fewer `num_key_value_heads` than `num_attention_heads`.

**TODO A.4:** write `kv_bytes_per_token` for ordinary full-attention layers.

In [ ]:
def kv_bytes_per_token(n_layers, n_kv_heads, head_dim, bytes_per_value=2):
    """KV-cache bytes for ONE token across n_layers full-attention layers (16-bit cache by default).
    Example: kv_bytes_per_token(32, 8, 128) -> 131072
    """
    # TODO: 2 * n_layers * n_kv_heads * head_dim * bytes_per_value
    return None


print(kv_bytes_per_token(32, 8, 128))

The helper below applies your function per layer type. `linear_attention` layers keep a fixed-size state instead of a growing cache, and `sliding_attention` layers never keep more than 1,024 tokens. It also shows what the cache would be if *every* layer were full attention, for comparison.

In [ ]:
# ---- Helper: KV cache for a given context length (just run) ----
CONTEXT = 32_768   # the course gateway caps requests at about 32K tokens


def kv_cache_gb(cfg, context=CONTEXT, all_full=False):
    total = 0
    for lt in cfg.get("layer_types") or ["full_attention"] * cfg["num_hidden_layers"]:
        if lt == "linear_attention" and not all_full:
            continue                                        # fixed-size state, does not grow
        if lt == "full_attention" and "global_head_dim" in cfg:    # Gemma global layers
            per_tok = kv_bytes_per_token(1, cfg["num_global_key_value_heads"], cfg["global_head_dim"])
            if cfg.get("attention_k_eq_v"):
                per_tok //= 2                               # keys double as values
        else:
            per_tok = kv_bytes_per_token(1, cfg["num_key_value_heads"], cfg["head_dim"])
        tokens = min(context, cfg.get("sliding_window", context)) if (lt == "sliding_attention" and not all_full) else context
        total += per_tok * tokens
    return total / 1e9


kv = None
if kv_bytes_per_token(32, 8, 128) is not None:
    kv = pd.DataFrame({a: {
        "KV_KB_per_token(32K avg)": round(kv_cache_gb(c) * 1e9 / CONTEXT / 1024, 1),
        "KV_GB_at_32K": round(kv_cache_gb(c), 2),
        "KV_GB_if_all_full_attention": round(kv_cache_gb(c, all_full=True), 2),
    } for a, c in configs.items()}).T
    display(kv)
else:
    print("Complete kv_bytes_per_token first.")

In [ ]:
ck.check("A4_kv_formula", kv_bytes_per_token(32, 8, 128) == 131072, 2,
         "kv_bytes_per_token(32, 8, 128) should be 131072 = 2 x 32 x 8 x 128 x 2.")
ck.check("A4_kv_table", kv is not None and 0.5 <= kv.loc["qwen-moe", "KV_GB_at_32K"] <= 0.8
         and kv.loc["qwen-dense", "KV_GB_at_32K"] > kv.loc["qwen-moe", "KV_GB_at_32K"], 2,
         "qwen-moe should need about 0.67 GB of KV cache at 32K tokens, and qwen-dense more.")
if kv is not None:
    ck.record("qwen_moe_kv_GB_32K", float(kv.loc["qwen-moe", "KV_GB_at_32K"]))
    ck.record("qwen_dense_kv_GB_32K", float(kv.loc["qwen-dense", "KV_GB_at_32K"]))

# Part B - Applied lab: does active-parameter count predict speed?

Generating text happens one token at a time (Chapter 2, **"Sampling"**). For every new token the hardware must read every *active* weight from memory once. On a single request, that memory read, not the math, is usually the bottleneck. So a rough upper limit is:

`tokens per second  <=  memory bandwidth / bytes of active weights`

An M3 Ultra has about **819 GB/s** of memory bandwidth (a T4 has about 320 GB/s). The cell below turns your Part A numbers into a "speed limit" for each model on the cluster. Real speeds will be lower (software overhead, other students' requests) and sometimes higher than you expect (`qwen-moe` uses multi-token prediction, a form of speculative decoding that can produce more than one token per weight read; Chapter 9 explains it).

In [ ]:
BANDWIDTH_GB_S = 819
QUANT_BITS = {"qwen-moe": 8, "qwen-dense": 4, "qwen-small": 8}   # how the cluster stores each model
SPEED_MODELS = [MODELS["main"], MODELS["dense"], MODELS["small"]]

bound = {}
if params is not None and weight_gb(1, 8) is not None:
    for m in SPEED_MODELS:
        active_gb = weight_gb(params.loc[m, "active_B"], QUANT_BITS[m])
        bound[m] = BANDWIDTH_GB_S / active_gb
        print(f"{m:11} active weights ~{active_gb:5.1f} GB  ->  upper bound ~{bound[m]:5.0f} tokens/s")
else:
    print("Complete Part A first.")

## B.1 Measure

You will send the same five Larkspur prompts to three models, twice each (30 requests plus 3 warm-up requests, about 3-6 minutes; `qwen-dense` is the slow one). The requests are interleaved so that a busy moment on the cluster does not hit only one model, and there is a short pause between calls to stay well under the rate limit.

We measure from the API response:
- **latency**: wall-clock seconds for the whole request
- **output tokens/sec**: `usage.completion_tokens / latency`. This slightly underestimates pure generation speed because latency includes reading the prompt (prefill) and network time. Chapter 9 separates these properly with streaming; today's rough number is enough.

**TODO B.1:** finish `measure()` (two lines).

In [ ]:
PROMPTS = [
    "Write a 120-word email to a Larkspur client explaining why their laptop needs a BIOS update tonight.",
    "Explain to a non-technical office manager, in about 120 words, what multi-factor authentication is.",
    "List eight practical steps a help-desk technician should take when a user says the printer is offline.",
    "In about 120 words, describe what a managed IT services provider does for a 20-person law firm.",
    "Write a short, friendly internal note (about 120 words) reminding technicians to log their time on tickets.",
]


def measure(model, prompt, max_tokens=300):
    """Send one request and return a dict with latency and output tokens/sec."""
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens, temperature=0.7, extra_body=NO_THINK)
    latency = time.perf_counter() - t0
    # TODO: get the number of generated tokens from resp.usage.completion_tokens
    out_tokens = None
    # TODO: output tokens per second = out_tokens / latency
    tps = None
    return {"model": model, "latency_s": latency, "out_tokens": out_tokens, "tps": tps,
            "text": resp.choices[0].message.content}


print(measure(MODELS["small"], "Say hello in five words.", max_tokens=20))

Now run the experiment (just run; it prints progress). If a request fails, it is recorded and the loop continues.

In [ ]:
REPEATS = 2
rows = []
for m in SPEED_MODELS:                                    # warm-up: first call can include model loading
    try:
        measure(m, "Reply with OK.", max_tokens=5)
    except Exception as e:
        print("warm-up failed for", m, e)

for rep in range(REPEATS):
    for i, prompt in enumerate(PROMPTS):
        for m in SPEED_MODELS:
            try:
                r = measure(m, prompt)
                r.update(rep=rep, prompt_id=i, error="")
            except Exception as e:
                r = {"model": m, "rep": rep, "prompt_id": i, "error": str(e)[:200]}
            rows.append(r)
            tps_txt = f"{r['tps']:.1f}" if isinstance(r.get("tps"), (int, float)) else "?"
            print(f"rep {rep} prompt {i} {m:11} latency={r.get('latency_s', 0):5.1f}s  tok/s={tps_txt}")
            time.sleep(1.0)

df = pd.DataFrame(rows)
print("\nrequests:", len(df), " errors:", int((df["error"] != "").sum()))

## B.2 Summarize and plot

**TODO B.2:** make a `summary` table with one row per model and the **median** `latency_s`, `out_tokens`, and `tps`. One line with `groupby` does it. Then compute the MoE-vs-dense speedup.

In [ ]:
ok_rows = df[df["error"] == ""].copy() if "error" in df else df

# TODO: summary = ok_rows.groupby("model")[["latency_s", "out_tokens", "tps"]].median()
summary = None

# TODO: speedup = median tps of qwen-moe divided by median tps of qwen-dense (use summary.loc[...])
speedup_moe_vs_dense = None

if summary is not None:
    if bound:
        summary["upper_bound_tps"] = pd.Series(bound)
        summary["percent_of_bound"] = (100 * summary["tps"] / summary["upper_bound_tps"]).round(0)
    display(summary.round(2))
print("MoE vs dense speedup:", speedup_moe_vs_dense)

In [ ]:
# ---- Plot (just run) ----
if summary is not None and params is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    order = [m for m in SPEED_MODELS if m in summary.index]
    axes[0].bar(order, [summary.loc[m, "tps"] for m in order], color="#4C72B0")
    axes[0].set_ylabel("median output tokens / second")
    axes[0].set_title("Measured generation speed")
    for m in order:
        x = params.loc[m, "active_B"]
        axes[1].scatter(x, summary.loc[m, "tps"], s=80, label=f"{m} measured")
        if m in bound:
            axes[1].scatter(x, bound[m], marker="x", s=80, color="gray")
        axes[1].annotate(m, (x, summary.loc[m, "tps"]), textcoords="offset points", xytext=(5, 5))
    axes[1].set_xlabel("active parameters (billions)")
    axes[1].set_ylabel("tokens / second")
    axes[1].set_title("Speed vs active parameters (x = bandwidth upper bound)")
    plt.tight_layout()
    plt.show()
else:
    print("Complete the TODOs above first.")

In [ ]:
n_ok = len(ok_rows)
ck.check("B1_measure_returns_tps", "tps" in ok_rows and ok_rows["tps"].notna().all() and n_ok > 0
         and (ok_rows["tps"] > 0).all(), 2,
         "measure() must fill out_tokens from resp.usage.completion_tokens and tps = out_tokens / latency.")
ck.check("B1_enough_requests", n_ok >= 27, 2,
         "At least 27 of 30 requests should succeed. Check the VPN, then re-run the experiment cell.")
ck.check("B2_summary_all_models", summary is not None and set(SPEED_MODELS) <= set(summary.index), 2,
         "summary needs one row per model: ok_rows.groupby('model')[[...]].median()")
ck.check("B2_moe_faster_than_dense", speedup_moe_vs_dense is not None and speedup_moe_vs_dense > 1.0, 2,
         "qwen-moe should generate faster than qwen-dense. Check speedup = moe tps / dense tps (not the reverse).")
if summary is not None:
    for m in SPEED_MODELS:
        if m in summary.index:
            ck.record(f"{m}_median_tps", float(summary.loc[m, "tps"]))
            ck.record(f"{m}_median_latency_s", float(summary.loc[m, "latency_s"]))
if speedup_moe_vs_dense is not None:
    ck.record("speedup_moe_vs_dense", float(speedup_moe_vs_dense))

# Written analysis

Answer each question in the box under it (3-6 sentences each). Use **your own numbers** from this notebook, and cite the chapter by section name where asked. Graded by the instructor (40 points).

**Q1. Total versus active parameters.** From your Part A table, how many parameters does `qwen-moe` store and how many does it use per token? What fraction is that? Using Chapter 2's discussion of sparse and mixture-of-experts models (section **"Model Size"**, including the Mixtral example), explain why a MoE model needs memory for its *total* parameters but costs compute for only its *active* parameters.

*Your answer:*

**Q2. Speed and memory bandwidth.** Report your measured median tokens/sec for the three models and the MoE-vs-dense speedup. Compare the speedup to what the ratio of active-weight bytes predicts (from the "upper bound" cell). Give at least two concrete reasons your measurements differ from the bound.

*Your answer:*

**Q3. Hardware fit.** Using your weight-memory and KV-cache tables, (a) check that the instructor's layout (8-bit `qwen-moe` alone on a 96 GB box; 4-bit `qwen-dense` sharing a box with the other models) fits under about 72 GB per box, including KV cache for a few 32K-token requests; and (b) say which of the four models you could run on a free Colab T4 (15 GB), at what precision, and how much context you would have left. Why does the hybrid attention design (linear or sliding layers) matter for a cluster serving ~10 students at once?

*Your answer:*

**Q4. Scaling laws.** Chapter 2 (**"Scaling law: Building compute-optimal models"**) describes the Chinchilla finding of roughly 20 training tokens per parameter. Compute the Chinchilla-optimal number of training tokens for `qwen-small` (4B) and `qwen-dense` (27B). Qwen reported pretraining its Qwen3 models on about 36 trillion tokens. Why would a model developer train far past the "compute-optimal" point? Connect your answer to what you measured about inference speed and cost, and to the chapter's discussion of scaling bottlenecks (training data and electricity).

*Your answer:*

**Q5. A deployment recommendation for Larkspur.** Larkspur's help-desk assistant must answer technicians quickly during business hours, often with several requests at once. Based on your measurements and memory math, would you recommend a dense model or a MoE model of similar total size? State one trade-off that goes against your recommendation (think about what the chapter says about how model size relates to quality, and about memory).

*Your answer:*

# AI-use reflection (10 points)

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. AI assistants often state parameter counts or memory figures confidently. Did you get a number from an AI that disagreed with your own calculation from the config? Which one did you trust, and why?

*Your answer:*

## Submit
Run this last, save, upload the .ipynb.

In [ ]:
ck.report()